# 02. ResNet50 + Clustering + Cluster-preserving QAT

공통 baseline에서 시작해 Conv2D / Dense layer에 clustering을 적용하고,
cluster-preserving QAT 후 TFLite로 변환합니다.

산출물:
- `RPS_ResNet50_Clustering_QAT.tflite`


## 필요 패키지

Jetson Thor 환경에 이미 설치되어 있으면 아래 셀은 실행하지 않아도 됩니다.

TensorFlow Model Optimization Toolkit이 없을 때만 `%pip install tensorflow-model-optimization`을 실행하세요.


In [1]:
# 필요할 때만 주석을 풀고 실행하세요.
%pip install -q tensorflow-model-optimization


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import gc
import glob
import zipfile
from pathlib import Path

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2
from sklearn.model_selection import train_test_split

import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.python.core.keras.compat import keras

print("TensorFlow:", tf.__version__)
print("TFMOT:", tfmot.__version__)
print("NumPy:", np.__version__)

# GPU 메모리를 처음부터 전부 잡지 않도록 설정합니다.
# Jetson에서 OOM을 줄이는 데 도움이 됩니다.
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as e:
        print("memory_growth 설정 생략:", e)

# 공통 설정
IMG_SIZE = 64
NUM_CLASSES = 3
SEED = 123

# OOM이 나면 가장 먼저 BATCH_SIZE를 4 또는 2로 낮추세요.
BATCH_SIZE = 8

# Jetson Thor에서 데이터셋 위치가 다르면 여기를 수정하세요.
# 또는 터미널에서 export RPS_DATASET_DIR=/your/path/RPS_Dataset 으로 지정해도 됩니다.
DEFAULT_DATASET_DIR = "/home/jaehun/jh/SYS_lecture/SYS5284/OnDeviceAI-Lightweighting-Lab/files/RPS_Dataset"
DATASET_DIR = Path(os.environ.get("RPS_DATASET_DIR", DEFAULT_DATASET_DIR))

# Colab에서 실행할 경우의 기본 경로도 자동 확인합니다.
if not DATASET_DIR.exists() and Path("/content/RPS_Dataset").exists():
    DATASET_DIR = Path("/content/RPS_Dataset")

# 결과 저장 폴더
SAVE_DIR = Path(os.environ.get("RPS_SAVE_DIR", str(Path.cwd() / "rps_outputs")))
SAVE_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_MODEL_PATH = SAVE_DIR / "baseline_resnet50_float.h5"
CACHE_NPZ_PATH = SAVE_DIR / f"rps_dataset_cache_{IMG_SIZE}.npz"

print("DATASET_DIR:", DATASET_DIR)
print("SAVE_DIR:", SAVE_DIR)
print("BASELINE_MODEL_PATH:", BASELINE_MODEL_PATH)


TensorFlow: 2.21.0
TFMOT: 0.8.1
NumPy: 2.4.6
GPUs: []
DATASET_DIR: /home/jaehun/jh/SYS_lecture/SYS5284/OnDeviceAI-Lightweighting-Lab/files/RPS_Dataset
SAVE_DIR: /home/jaehun/jh/SYS_lecture/SYS5284/OnDeviceAI-Lightweighting-Lab/notebooks_jh/rps_outputs
BASELINE_MODEL_PATH: /home/jaehun/jh/SYS_lecture/SYS5284/OnDeviceAI-Lightweighting-Lab/notebooks_jh/rps_outputs/baseline_resnet50_float.h5


In [3]:
BATCH_SIZE = 2  # conv5_까지 경량화하면 메모리 사용량이 커지므로 2부터 권장합니다.

# 용량 차이가 보이도록 ResNet50 마지막 stage(conv5_)와 Dense 분류기를 대상으로 삼습니다.
# OOM이 나면 *_CONV2D_LAYERS를 False로 낮추거나 BATCH_SIZE=1로 낮추세요.
TARGET_RESNET_STAGES = ("conv5_",)
CLUSTER_CONV2D_LAYERS = True
QAT_CONV2D_LAYERS = True


In [4]:
def load_rps_dataset(dataset_dir, img_size=64, seed=123):
    # RPS_Dataset/
    #   0/ : scissors
    #   1/ : rock
    #   2/ : paper
    # 구조를 읽어 train/test numpy 배열로 변환합니다.
    dataset_dir = Path(dataset_dir)
    if not dataset_dir.exists():
        raise FileNotFoundError(f"DATASET_DIR가 존재하지 않습니다: {dataset_dir}")

    first = True

    for ind in range(NUM_CLASSES):
        pattern = str(dataset_dir / str(ind) / "*.*")
        files = sorted(glob.glob(pattern))
        print(f"class {ind}: {len(files)} files")

        if len(files) == 0:
            raise FileNotFoundError(f"이미지를 찾지 못했습니다: {pattern}")

        images = []
        for f in files:
            img = cv2.imread(f, cv2.IMREAD_COLOR)
            if img is None:
                print("이미지 읽기 실패:", f)
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (img_size, img_size))
            images.append(img)

        tmpx = np.array(images, dtype=np.uint8)
        tmpy = np.array([ind] * len(tmpx), dtype=np.int64)

        xtrain, xtest, ytrain, ytest = train_test_split(
            tmpx, tmpy, test_size=0.2, random_state=seed, stratify=tmpy
        )

        if first:
            train_data = xtrain
            train_labels = ytrain
            test_data = xtest
            test_labels = ytest
            first = False
        else:
            train_data = np.concatenate((train_data, xtrain), axis=0)
            train_labels = np.concatenate((train_labels, ytrain), axis=0)
            test_data = np.concatenate((test_data, xtest), axis=0)
            test_labels = np.concatenate((test_labels, ytest), axis=0)

    return train_data, train_labels, test_data, test_labels


def prepare_data():
    # 첫 실행 때는 이미지 폴더에서 데이터를 읽고,
    # 이후 실행부터는 npz 캐시를 사용해 시간을 줄입니다.
    if CACHE_NPZ_PATH.exists():
        print("캐시 데이터 로드:", CACHE_NPZ_PATH)
        data = np.load(CACHE_NPZ_PATH)
        train_data = data["train_data"]
        train_labels = data["train_labels"]
        test_data = data["test_data"]
        test_labels = data["test_labels"]
    else:
        print("이미지 폴더에서 데이터 로드:", DATASET_DIR)
        train_data, train_labels, test_data, test_labels = load_rps_dataset(
            DATASET_DIR, IMG_SIZE, SEED
        )
        np.savez_compressed(
            CACHE_NPZ_PATH,
            train_data=train_data,
            train_labels=train_labels,
            test_data=test_data,
            test_labels=test_labels
        )
        print("캐시 저장:", CACHE_NPZ_PATH)

    print("train_data:", train_data.shape, train_data.dtype)
    print("train_labels:", train_labels.shape)
    print("test_data:", test_data.shape, test_data.dtype)
    print("test_labels:", test_labels.shape)

    return train_data, train_labels, test_data, test_labels


train_data, train_labels, test_data, test_labels = prepare_data()

# 데이터 보강
# 기존 rotation_range=0.2는 0.2도가 되어 거의 회전하지 않습니다.
# 여기서는 20도로 수정했습니다.
img_gen_train = tf.keras.preprocessing.image.ImageDataGenerator(
    horizontal_flip=True,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=(0.6, 1.4),
    zoom_range=0.2
)

train_gen = img_gen_train.flow(
    train_data,
    train_labels,
    batch_size=BATCH_SIZE,
    seed=SEED,
    shuffle=True
)


이미지 폴더에서 데이터 로드: /home/jaehun/jh/SYS_lecture/SYS5284/OnDeviceAI-Lightweighting-Lab/files/RPS_Dataset
class 0: 903 files
class 1: 907 files
class 2: 907 files
캐시 저장: /home/jaehun/jh/SYS_lecture/SYS5284/OnDeviceAI-Lightweighting-Lab/notebooks_jh/rps_outputs/rps_dataset_cache_64.npz
train_data: (2172, 64, 64, 3) uint8
train_labels: (2172,)
test_data: (545, 64, 64, 3) uint8
test_labels: (545,)


In [5]:
def clear_memory():
    gc.collect()
    tf.keras.backend.clear_session()


def compile_for_rps(model_to_compile, learning_rate=1e-5):
    model_to_compile.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"]
    )
    return model_to_compile


def evaluate_rps_model(model_to_eval, title="model"):
    if getattr(model_to_eval, "optimizer", None) is None:
        compile_for_rps(model_to_eval)
    loss, acc = model_to_eval.evaluate(test_data, test_labels, verbose=0)
    print(f"{title} loss: {loss:.4f}, accuracy: {acc:.4f}")
    return loss, acc


def is_trainable_weight_layer(layer):
    return isinstance(layer, (keras.layers.Conv2D, keras.layers.Dense)) and len(layer.weights) > 0


def should_transform_layer(layer, include_conv2d=False):
    if isinstance(layer, keras.layers.Dense):
        return len(layer.weights) > 0

    if include_conv2d and isinstance(layer, keras.layers.Conv2D):
        target_stages = globals().get("TARGET_RESNET_STAGES", ("conv5_",))
        return len(layer.weights) > 0 and layer.name.startswith(target_stages)

    return False


def apply_quantization_to_layer(layer):
    # ResNet50 전체 Conv2D를 QAT로 감싸면 메모리 사용량이 크게 증가합니다.
    # 기본은 Dense 분류기만 QAT 대상으로 삼고, 필요하면 QAT_CONV2D_LAYERS=True로 바꿉니다.
    if should_transform_layer(layer, include_conv2d=QAT_CONV2D_LAYERS):
        return tfmot.quantization.keras.quantize_annotate_layer(layer)
    return layer


def annotate_conv_dense_for_qat(model_to_annotate):
    return keras.models.clone_model(
        model_to_annotate,
        clone_function=apply_quantization_to_layer
    )


def representative_dataset():
    # TFLite 변환 시 양자화 scale 추정을 위한 대표 데이터셋입니다.
    # 너무 많이 넣으면 변환 시간이 길어지므로 100개만 사용합니다.
    n = min(100, len(train_data))
    for i in range(n):
        sample = train_data[i:i+1].astype(np.float32)
        yield [sample]


def zip_tflite_file(tflite_path):
    # pruning/clustering 효과는 원본 .tflite보다 zip 압축 크기에서 더 잘 보일 수 있습니다.
    # 비교용으로 .zip도 같이 저장합니다.
    tflite_path = Path(tflite_path)
    zip_path = tflite_path.with_suffix(tflite_path.suffix + ".zip")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(tflite_path, arcname=tflite_path.name)
    print("ZIP saved:", zip_path, zip_path.stat().st_size, "bytes")
    return zip_path


def save_as_tflite(model_to_convert, tflite_file_name):
    # 기본은 float input/output을 유지한 TFLite 최적화 변환입니다.
    # 라즈베리파이에서 전처리 코드를 크게 바꾸지 않아도 실행하기 쉽습니다.
    tflite_path = SAVE_DIR / tflite_file_name

    converter = tf.lite.TFLiteConverter.from_keras_model(model_to_convert)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset

    # 완전 int8 모델이 필요하면 아래 3줄의 주석을 해제하세요.
    # 단, 라즈베리파이 추론 코드에서 input/output dtype 처리가 달라질 수 있습니다.
    # converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    # converter.inference_input_type = tf.uint8
    # converter.inference_output_type = tf.uint8

    tflite_model = converter.convert()
    tflite_path.write_bytes(tflite_model)
    print("TFLite saved:", tflite_path, tflite_path.stat().st_size, "bytes")
    zip_tflite_file(tflite_path)
    return tflite_path


In [6]:
if not BASELINE_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"baseline 모델이 없습니다: {BASELINE_MODEL_PATH}\n"
        "먼저 00_train_baseline_resnet50.ipynb를 실행해서 baseline_resnet50_float.h5를 생성하세요."
    )

baseline_model = keras.models.load_model(BASELINE_MODEL_PATH, compile=False)
baseline_model.summary()
evaluate_rps_model(baseline_model, "Loaded baseline")


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 64, 64, 3)]          0         []                            
                                                                                                  
 tf.__operators__.getitem (  (None, 64, 64, 3)            0         ['input_1[0][0]']             
 SlicingOpLambda)                                                                                 
                                                                                                  
 tf.nn.bias_add (TFOpLambda  (None, 64, 64, 3)            0         ['tf.__operators__.getitem[0][
 )                                                                  0]']                          
                                                                                              

(0.2795056402683258, 0.9339449405670166)

In [7]:
CLUSTER_FINE_TUNE_EPOCHS = 3
QAT_EPOCHS = 1

# 이 셀만 단독 실행해도 conv5_ 대상 설정이 유지되도록 기본 옵션을 보강합니다.
TARGET_RESNET_STAGES = globals().get("TARGET_RESNET_STAGES", ("conv5_",))
CLUSTER_CONV2D_LAYERS = globals().get("CLUSTER_CONV2D_LAYERS", True)
QAT_CONV2D_LAYERS = globals().get("QAT_CONV2D_LAYERS", True)


clustering_params = {
    "number_of_clusters": 4,
    "cluster_centroids_init": tfmot.clustering.keras.CentroidInitialization.KMEANS_PLUS_PLUS,
}

# per-channel clustering은 ResNet50에서 메모리 사용량과 skip 경고가 커서 사용하지 않습니다.

# 로드된 모델을 한 번 호출해 build/weight 상태를 확정합니다.
_ = baseline_model(test_data[:1], training=False)


def apply_clustering_to_layer(layer):
    if should_transform_layer(layer, include_conv2d=CLUSTER_CONV2D_LAYERS):
        try:
            return tfmot.clustering.keras.cluster_weights(layer, **clustering_params)
        except ValueError as e:
            print(f"Skip clustering for {layer.name}: {e}")
            return layer
    return layer

clustered_model = keras.models.clone_model(
    baseline_model,
    clone_function=apply_clustering_to_layer
)

compile_for_rps(clustered_model, learning_rate=1e-5)
clustered_model.summary()

history_cluster = clustered_model.fit(
    train_gen,
    epochs=CLUSTER_FINE_TUNE_EPOCHS,
    validation_data=(test_data, test_labels)
)

stripped_clustered_model = tfmot.clustering.keras.strip_clustering(clustered_model)
evaluate_rps_model(stripped_clustered_model, "Stripped clustered model")

del baseline_model, clustered_model
gc.collect()


Skip clustering for conv5_block1_1_conv: Layer conv5_block1_1_conv has no weights to cluster.
Skip clustering for conv5_block1_2_conv: Layer conv5_block1_2_conv has no weights to cluster.
Skip clustering for conv5_block1_0_conv: Layer conv5_block1_0_conv has no weights to cluster.
Skip clustering for conv5_block1_3_conv: Layer conv5_block1_3_conv has no weights to cluster.
Skip clustering for conv5_block2_1_conv: Layer conv5_block2_1_conv has no weights to cluster.
Skip clustering for conv5_block2_2_conv: Layer conv5_block2_2_conv has no weights to cluster.
Skip clustering for conv5_block2_3_conv: Layer conv5_block2_3_conv has no weights to cluster.
Skip clustering for conv5_block3_1_conv: Layer conv5_block3_1_conv has no weights to cluster.
Skip clustering for conv5_block3_2_conv: Layer conv5_block3_2_conv has no weights to cluster.
Skip clustering for conv5_block3_3_conv: Layer conv5_block3_3_conv has no weights to cluster.
Model: "model"
_____________________________________________

I0000 00:00:1782053002.082961  583455 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.


1086/1086 [==============================] - 54s 48ms/step - loss: 0.4437 - accuracy: 0.8287 - val_loss: 0.2763 - val_accuracy: 0.9101
Epoch 2/3
1086/1086 [==============================] - 53s 49ms/step - loss: 0.4055 - accuracy: 0.8462 - val_loss: 0.2660 - val_accuracy: 0.9193
Epoch 3/3
1086/1086 [==============================] - 51s 47ms/step - loss: 0.4192 - accuracy: 0.8416 - val_loss: 0.2583 - val_accuracy: 0.9248
Stripped clustered model loss: 0.2583, accuracy: 0.9248


18164

In [8]:
# 이 셀만 단독 실행해도 conv5_ QAT 대상 설정이 유지되도록 기본 옵션을 보강합니다.
TARGET_RESNET_STAGES = globals().get("TARGET_RESNET_STAGES", ("conv5_",))
QAT_CONV2D_LAYERS = globals().get("QAT_CONV2D_LAYERS", True)

annotated_clustered_model = annotate_conv_dense_for_qat(stripped_clustered_model)

cqat_model = tfmot.quantization.keras.quantize_apply(
    annotated_clustered_model,
    tfmot.experimental.combine.Default8BitClusterPreserveQuantizeScheme()
)

compile_for_rps(cqat_model, learning_rate=1e-5)
cqat_model.summary()

history_qat = cqat_model.fit(
    train_gen,
    epochs=QAT_EPOCHS,
    validation_data=(test_data, test_labels)
)

evaluate_rps_model(cqat_model, "Clustering + QAT")


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 64, 64, 3)]          0         []                            
                                                                                                  
 tf.__operators__.getitem (  (None, 64, 64, 3)            0         ['input_1[0][0]']             
 SlicingOpLambda)                                                                                 
                                                                                                  
 tf.nn.bias_add (TFOpLambda  (None, 64, 64, 3)            0         ['tf.__operators__.getitem[1][
 )                                                                  0]']                          
                                                                                              

1086/1086 [==============================] - 93s 83ms/step - loss: 0.2895 - accuracy: 0.8844 - val_loss: 0.1714 - val_accuracy: 0.9541
Clustering + QAT loss: 0.1714, accuracy: 0.9541


(0.1713523119688034, 0.9541284441947937)

In [9]:
cqat_model.save(SAVE_DIR / "RPS_ResNet50_Clustering_QAT_model.h5", include_optimizer=False)
save_as_tflite(cqat_model, "RPS_ResNet50_Clustering_QAT.tflite")

del stripped_clustered_model, annotated_clustered_model, cqat_model
clear_memory()
print("Clustering + QAT notebook done.")


/home/jaehun/.venv/lib/python3.12/site-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


INFO:tensorflow:Assets written to: /tmp/tmphx6mv70z/assets


INFO:tensorflow:Assets written to: /tmp/tmphx6mv70z/assets
/home/jaehun/.venv/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1782053375.418869  583455 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1782053375.418980  583455 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1782053375.419863  583455 reader.cc:83] Reading SavedModel from: /tmp/tmphx6mv70z
I0000 00:00:1782053375.497471  583455 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1782053375.497564  583455 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmphx6mv70z
I0000 00:00:1782053375.708366  583455 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1782053375.762149  583455 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1782053376.831822  583455

TFLite saved: /home/jaehun/jh/SYS_lecture/SYS5284/OnDeviceAI-Lightweighting-Lab/notebooks_jh/rps_outputs/RPS_ResNet50_Clustering_QAT.tflite 24239392 bytes
ZIP saved: /home/jaehun/jh/SYS_lecture/SYS5284/OnDeviceAI-Lightweighting-Lab/notebooks_jh/rps_outputs/RPS_ResNet50_Clustering_QAT.tflite.zip 20254694 bytes
Clustering + QAT notebook done.
